# Module 3 - Detecting Crop Changes Over Time

In [ ]:
# Core scientific computing
import numpy as np
import json
import io
import base64

# Geospatial data handling
import rasterio
from rasterio.features import rasterize
from shapely.geometry import shape, box
from shapely.ops import transform
import pyproj

# Visualization
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import folium
from PIL import Image

# Deep learning with PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Check if GPU is available for faster training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### 3.1. NDVI Change Detection (Baseline)


#### Load and Visualize Both Dates


In [ ]:
# Load both processed images
with rasterio.open('april_processed.tif') as src:
    image_april = np.transpose(src.read(), (1, 2, 0))
    shape_raster = (src.height, src.width)
    transform = src.transform
    bounds = src.bounds
    
with rasterio.open('july_processed.tif') as src:
    image_july = np.transpose(src.read(), (1, 2, 0))


print(f"✓ Loaded images")
print(f"  April: {image_april.shape}")
print(f"  July: {image_july.shape}")

In [ ]:
# Visualize both dates side-by-side
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].imshow(image_april)
axes[0].set_title('April 2025\n(Early Season - More Bare Soil)', 
                  fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(image_july)
axes[1].set_title('July 2025\n(Mid Season - More Vegetation)', 
                  fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.suptitle('Multi-Temporal Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

#### Calculate NDVI for Both Dates


**KEY CONCEPTS:**
- **True NDVI** = uses NIR and Red bands from raw Sentinel-2 data
- **RGB NDVI proxy** = (Green - Red) / (Green + Red), approximation for demo purposes
- **NDVI range** = -1 to +1, where >0.5 is dense vegetation, <0.2 is bare soil


In [ ]:
def calculate_ndvi_rgb(image):
    """
    Calculate NDVI approximation from RGB imagery
    
    Uses green-red difference as proxy for vegetation
    Range: -1 to +1
      - High (>0.5): Dense vegetation
      - Medium (0.2-0.5): Sparse vegetation  
      - Low (<0.2): Bare soil, water
    """
    red = image[:, :, 0]
    green = image[:, :, 1]
    
    # NDVI proxy using visible bands only
    ndvi = (green - red) / (green + red + 1e-6)
    
    return ndvi

# Calculate NDVI for both dates
ndvi_april = calculate_ndvi_rgb(image_april)
ndvi_july = calculate_ndvi_rgb(image_july)

print(f"✓ NDVI calculated")
print(f"  April NDVI range: [{ndvi_april.min():.3f}, {ndvi_april.max():.3f}]")
print(f"  July NDVI range: [{ndvi_july.min():.3f}, {ndvi_july.max():.3f}]")

In [ ]:
# Create simple side-by-side visualization
fig = plt.figure(figsize=(12, 4))

# Panel 1: NDVI April
plt.subplot(1, 2, 1)
plt.imshow(ndvi_april, cmap='RdYlGn', vmin=-0.5, vmax=1.0)
plt.colorbar(fraction=0.046, pad=0.04)
plt.title('NDVI - April 2025', fontsize=13, fontweight='bold')
plt.axis('off')

# Panel 2: NDVI July
plt.subplot(1, 2, 2)
plt.imshow(ndvi_july, cmap='RdYlGn', vmin=-0.5, vmax=1.0)
plt.colorbar(fraction=0.046, pad=0.04)
plt.title('NDVI - July 2025', fontsize=13, fontweight='bold')
plt.axis('off')

plt.suptitle('NDVI Comparison: April vs July 2025',
             fontsize=15, fontweight='bold')

plt.tight_layout()
plt.show()


#### Compute NDVI Change


In [ ]:
# Calculate NDVI change (July - April)
# Positive = vegetation increased (planting)
# Negative = vegetation decreased (harvest)
ndvi_change = ndvi_july - ndvi_april

# Define change threshold
change_threshold = 0.15

# Categorize pixels by change magnitude
increase = ndvi_change > change_threshold       # Vegetation gain
decrease = ndvi_change < -change_threshold      # Vegetation loss
stable = np.abs(ndvi_change) <= change_threshold  # No significant change

print(f"✓ Change detection complete")
print(f"\nChange statistics:")
print(f"  Vegetation increase: {increase.sum():,} pixels ({increase.sum()/increase.size*100:.1f}%)")
print(f"  Vegetation decrease: {decrease.sum():,} pixels ({decrease.sum()/decrease.size*100:.1f}%)")
print(f"  Stable (no change):  {stable.sum():,} pixels ({stable.sum()/stable.size*100:.1f}%)")

#### Visualize Change Detection Results


In [ ]:
# Create simplified visualization
fig = plt.figure(figsize=(12, 4))

# -------------------------
# Panel 1: NDVI Change
# -------------------------
plt.subplot(1, 2, 1)
im = plt.imshow(ndvi_change, cmap='RdBu_r', vmin=-0.5, vmax=0.5)
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.title('NDVI Change (April → July)\nRed = Loss | Blue = Gain',
          fontsize=12, fontweight='bold')
plt.axis('off')

# -------------------------
# Panel 2: Discrete Change Categories
# -------------------------
plt.subplot(1, 2, 2)

change_map = np.zeros_like(ndvi_change)
change_map[increase] = 1    # Gain
change_map[decrease] = -1   # Loss
change_map[stable] = 0      # Stable

colors = ['#d62728', '#ffffff', '#2ca02c']  # Red, White, Green
cmap = ListedColormap(colors)

plt.imshow(change_map, cmap=cmap, vmin=-1, vmax=1)
plt.title('Change Categories\nRed=Loss | Green=Gain | White=Stable',
          fontsize=12, fontweight='bold')
plt.axis('off')

# Legend
legend_elements = [
    Patch(facecolor='#d62728', label='Vegetation Loss'),
    Patch(facecolor='#ffffff', edgecolor='black', label='Stable'),
    Patch(facecolor='#2ca02c', label='Vegetation Gain')
]
plt.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.suptitle('NDVI-Based Vegetation Change Detection',
             fontsize=15, fontweight='bold')

plt.tight_layout()
plt.show()

### 3.2. Field-Level Change Analysis

#### Load Field Boundaries 


In [ ]:
# Load field parcel boundaries from Module 2
with open('field_parcels.geojson', 'r') as f:
    fields_geojson = json.load(f)

n_fields = len(fields_geojson['features'])
print(f"✓ Loaded {n_fields} field parcels from Module 2")

In [ ]:
ndvi_april

In [ ]:
ndvi_july

#### Compute Zonal Statistics Per Field


In [ ]:
# Storage for field-level statistics
field_stats = []

for feature in fields_geojson['features']:
    field_id = feature['properties']['parcel_id']
    area_ha = feature['properties']['area_ha']

    # Rasterize this field to create a mask
    geom = shape(feature['geometry'])
    field_mask = rasterize(
        [(geom, 1)],
        out_shape=shape_raster,
        transform=transform,
        fill=0,
        dtype=np.uint8
    ).astype(bool)

     # Skip if field is outside raster bounds
    if not field_mask.any():
        continue
    
    # Calculate mean NDVI for this field in both dates
    ndvi_april_mean = ndvi_april[field_mask].mean()
    ndvi_july_mean = ndvi_july[field_mask].mean()
    ndvi_change_mean = ndvi_july_mean - ndvi_april_mean

    # Store statistics
    field_stats.append({
        'field_id': field_id,
        'area_ha': area_ha,
        'ndvi_april_mean': ndvi_april_mean,
        'ndvi_july_mean': ndvi_july_mean,
        'ndvi_change_mean': ndvi_change_mean
    })

print(f"✓ Processed {len(field_stats)} fields")
print(f"\nSample statistics:")
print(f"  Mean April NDVI: {np.mean([s['ndvi_april_mean'] for s in field_stats]):.3f}")
print(f"  Mean July NDVI: {np.mean([s['ndvi_july_mean'] for s in field_stats]):.3f}")
print(f"  Mean change: {np.mean([s['ndvi_change_mean'] for s in field_stats]):.3f}")    

In [ ]:
field_stats

In [ ]:
# Classify fields based on NDVI change magnitude
def classify_change_direct(change):
    """
    Classify based on NDVI change magnitude
    Positive = vegetation increase
    Negative = vegetation decrease
    """
    if change > 0.10:
        return 1, 'Vegetation increase'
    elif change < -0.10:
        return 2, 'Vegetation decrease'
    else:
        return 0, 'Stable'

# Classify fields based on NDVI change magnitude
def classify_change_direct(change):
    """
    Classify based on NDVI change magnitude
    Positive = vegetation increase
    Negative = vegetation decrease
    """
    if change > 0.10:
        return 1, 'Vegetation increase'
    elif change < -0.10:
        return 2, 'Vegetation decrease'
    else:
        return 0, 'Stable'


# Classify each field
for s in field_stats:
    s['change_type'], s['change_label'] = classify_change_direct(s['ndvi_change_mean'])


# Count fields in each category
change_counts = {}
for s in field_stats:
    change_counts[s['change_type']] = change_counts.get(s['change_type'], 0) + 1

# Labels
change_labels_simple = {
    0: 'Stable',
    1: 'Vegetation increase',
    2: 'Vegetation decrease'
}

print(f"\n✓ Field classification summary:")
for ct, count in sorted(change_counts.items()):
    pct = count / len(field_stats) * 100
    print(f"  {change_labels_simple[ct]:20s}: {count:3d} fields ({pct:5.1f}%)")

In [ ]:
# Summary statistics
print("\n" + "="*60)
print("FIELD-LEVEL CHANGE ANALYSIS COMPLETE")
print("="*60)

print(f"\n📊 Results:")
total_area = sum(s['area_ha'] for s in field_stats)
increase_area = sum(s['area_ha'] for s in field_stats if s['change_type'] == 1)
decrease_area = sum(s['area_ha'] for s in field_stats if s['change_type'] == 2)
stable_area = sum(s['area_ha'] for s in field_stats if s['change_type'] == 0)

print(f"   • Total area analyzed: {total_area:.1f} ha")
print(f"   • Vegetation increase: {increase_area:.1f} ha ({increase_area/total_area*100:.1f}%)")
print(f"   • Vegetation decrease: {decrease_area:.1f} ha ({decrease_area/total_area*100:.1f}%)")
print(f"   • Stable: {stable_area:.1f} ha ({stable_area/total_area*100:.1f}%)")

print(f"\n✅ Advantages over pixel-level:")
print("   • Practical units (fields, hectares)")
print("   • Reduces noise by aggregating")
print("   • Matches agricultural management units")


#### Interactive Map with Field-Level Changes


In [ ]:
# Get map center
min_lon, min_lat = bounds.left, bounds.bottom
max_lon, max_lat = bounds.right, bounds.top
center_lat = (min_lat + max_lat) / 2
center_lon = (min_lon + max_lon) / 2

# Create base map
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=14,
    tiles='OpenStreetMap'
)

# Add Sentinel-2 RGB overlay (July)
print("  Adding RGB imagery overlay...")
rgb_8bit = (np.clip(image_july, 0, 1) * 255).astype(np.uint8)
img = Image.fromarray(rgb_8bit)
buffer = io.BytesIO()
img.save(buffer, format='PNG')
img_str = base64.b64encode(buffer.getvalue()).decode()

folium.raster_layers.ImageOverlay(
    image=f'data:image/png;base64,{img_str}',
    bounds=[[min_lat, min_lon], [max_lat, max_lon]],
    opacity=0.7,
    name='Sentinel-2 July'
).add_to(m)

# Color mapping for change categories
change_colors_hex = {
    0: '#C8C8C8',  # Stable - gray
    1: '#228B22',  # Increase - green
    2: '#DC143C'   # Decrease - red
}

# Add field polygons with statistics
print("  Adding field boundaries...")
stats_dict = {s['field_id']: s for s in field_stats}

for feature in fields_geojson['features']:
    field_id = feature['properties']['parcel_id']
    
    if field_id not in stats_dict:
        continue
    
    stats = stats_dict[field_id]
    
    # Create popup with statistics
    popup_html = f"""
    <b>Field {field_id}</b><br>
    Area: {stats['area_ha']:.2f} ha<br>
    <br>
    <b>NDVI Values:</b><br>
    April: {stats['ndvi_april_mean']:.3f}<br>
    July: {stats['ndvi_july_mean']:.3f}<br>
    Change: {stats['ndvi_change_mean']:+.3f}<br>
    <br>
    <b>Classification:</b><br>
    {stats['change_label']}
    """
    
    # Get color based on change type
    color = change_colors_hex[stats['change_type']]
    
    # Add polygon
    folium.GeoJson(
        feature['geometry'],
        style_function=lambda x, c=color: {
            'fillColor': c,
            'color': 'black',
            'weight': 1,
            'fillOpacity': 0.5
        },
        popup=folium.Popup(popup_html, max_width=300)
    ).add_to(m)

# Add legend
print("  Adding legend...")
legend_html = f'''
<div style="position: fixed; 
            bottom: 50px; right: 50px; width: 220px; 
            background-color: white; z-index:9999; font-size:14px;
            border:2px solid grey; border-radius: 5px; padding: 10px">
<p style="margin: 0; font-weight: bold;">Field-Level Changes</p>
<p style="margin: 5px 0;"><span style="background-color: {change_colors_hex[0]}; padding: 5px;">&nbsp;&nbsp;&nbsp;</span> Stable</p>
<p style="margin: 5px 0;"><span style="background-color: {change_colors_hex[1]}; padding: 5px;">&nbsp;&nbsp;&nbsp;</span> Vegetation increase</p>
<p style="margin: 5px 0;"><span style="background-color: {change_colors_hex[2]}; padding: 5px;">&nbsp;&nbsp;&nbsp;</span> Vegetation decrease</p>
<hr style="margin: 5px 0;">
<p style="margin: 5px 0; font-size: 12px;"><b>Total fields:</b> {len(field_stats)}</p>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))


m

### 3.3 Siamese U-Net for Deep Learning Change Detection

**KEY CONCEPTS:**
- **Siamese architecture** = two identical encoders with shared weights process both input images
- **Weight sharing** = ensures both dates use same feature detectors for fair comparison


#### Prepare Training Data

In [ ]:
# Create binary change labels from NDVI change
# Threshold at ±0.10 to identify significant changes
change_binary = (np.abs(ndvi_change) > 0.10).astype(np.float32)

print(f"  Change pixels: {change_binary.sum():,} ({change_binary.mean()*100:.1f}%)")
print(f"  No-change pixels: {(1-change_binary).sum():,} ({(1-change_binary).mean()*100:.1f}%)")

In [ ]:
# Spatial split - same strategy as crop detection
h, w = image_april.shape[:2]
split_row = h // 2

april_train  = image_april[:split_row, :, :]
april_test   = image_april[split_row:, :, :]

july_train   = image_july[:split_row, :, :]
july_test    = image_july[split_row:, :, :]

change_train = change_binary[:split_row, :]
change_test  = change_binary[split_row:, :]


print(f"✓ Spatial split applied")
print(f"  Train (top half):  {april_train.shape}")
print(f"  Test (bottom half): {april_test.shape}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for row, (apr, jul, chg, label, color) in enumerate([
    (april_train, july_train, change_train, 'TRAIN (TOP - LEFT)', 'blue'),
    (april_test,  july_test,  change_test,  'TEST (BOTTOM - NEVER SEEN!)', 'red')
]):
    axes[row, 0].imshow(apr)
    axes[row, 0].set_title(f'April\n{label}', fontweight='bold', color=color)
    axes[row, 0].axis('off')
    
    axes[row, 1].imshow(jul)
    axes[row, 1].set_title(f'July\n{label}', fontweight='bold', color=color)
    axes[row, 1].axis('off')
    
    axes[row, 2].imshow(chg, cmap='RdYlGn_r', vmin=0, vmax=1)
    axes[row, 2].set_title(f'Change Labels\n{chg.sum():,} changed pixels', 
                           fontweight='bold', color=color)
    axes[row, 2].axis('off')

plt.suptitle('Spatial Split: Training (Blue) vs Test (Red)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Extract random patch pairs with labels
patch_size = 64
n_samples = 200

def extract_patch_pairs(img1, img2, labels, size, n):
    """Extract random patch pairs from two images with corresponding labels"""
    patches1, patches2, patch_labels = [], [], []
    h, w = img1.shape[:2]
    
    for _ in range(n):
        # Random location
        y = np.random.randint(0, h - size)
        x = np.random.randint(0, w - size)
        
        # Extract patches
        p1 = img1[y:y+size, x:x+size]
        p2 = img2[y:y+size, x:x+size]
        label = labels[y:y+size, x:x+size]
        
        # Convert to PyTorch tensors (C, H, W format)
        p1 = torch.FloatTensor(p1).permute(2, 0, 1)
        p2 = torch.FloatTensor(p2).permute(2, 0, 1)
        label = torch.FloatTensor(label).unsqueeze(0)
        
        patches1.append(p1)
        patches2.append(p2)
        patch_labels.append(label)
    
    return torch.stack(patches1), torch.stack(patches2), torch.stack(patch_labels)


# Extract patch pairs from LEFT HALF only
X1, X2, y = extract_patch_pairs(april_train, july_train, change_train, patch_size, n_samples)

print(f"✓ Extracted {n_samples} patch pairs (LEFT HALF only)")
print(f"  April patches: {X1.shape}")
print(f"  July patches:  {X2.shape}")
print(f"  Labels:        {y.shape}")
print(f"  Changed pixels: {y.float().mean()*100:.1f}%")

In [ ]:
# Show 4 random patch pair examples
fig, axes = plt.subplots(3, 4, figsize=(14, 9))

for i in range(4):
    idx = np.random.randint(0, len(X1))
    
    axes[0, i].imshow(X1[idx].permute(1, 2, 0))
    axes[0, i].set_title(f'April #{idx}', fontweight='bold', fontsize=10)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(X2[idx].permute(1, 2, 0))
    axes[1, i].set_title(f'July #{idx}', fontweight='bold', fontsize=10)
    axes[1, i].axis('off')
    
    axes[2, i].imshow(y[idx].squeeze(), cmap='RdYlGn_r', vmin=0, vmax=1)
    axes[2, i].set_title(f'Change Label', fontweight='bold', fontsize=10)
    axes[2, i].axis('off')

plt.suptitle('Sample Patch Pairs: April → July → Change Label', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

#### Define Siamese U-Net Architecture


**KEY CONCEPTS:**
- **Shared encoder** = one encoder used twice (not two separate encoders)
- **Feature concatenation** = combining April and July features to capture temporal differences
- **Bottleneck** = compressed representation that processes combined temporal information
- **Decoder** = upsampling back to original resolution for pixel-wise output


In [ ]:
class SiameseUNet(nn.Module):
    """
    Siamese U-Net for change detection
    
    Architecture:
    - Shared encoder processes both dates identically
    - Concatenate features from both dates
    - Decoder produces pixel-wise change probability map
    """
    
    def __init__(self, in_channels=3):
        super(SiameseUNet, self).__init__()
        
        # Shared encoder (downsampling)
        self.enc1 = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU()
        )
        self.pool1 = nn.MaxPool2d(2)  # 64×64 → 32×32
        
        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU()
        )
        self.pool2 = nn.MaxPool2d(2)  # 32×32 → 16×16
        
        # Bottleneck (processes concatenated features from both dates)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(128, 128, 3, padding=1),  # 64*2 from concatenation
            nn.ReLU()
        )
        
        # Decoder (upsampling)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)  # 16×16 → 32×32
        self.dec1 = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU()
        )
        
        self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2)  # 32×32 → 64×64
        self.dec2 = nn.Sequential(
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU()
        )
        
        # Output layer (change probability)
        self.output = nn.Conv2d(32, 1, 1)
        self.sigmoid = nn.Sigmoid()
    
    def encode(self, x):
        """Encode one image through shared encoder"""
        e1 = self.enc1(x)
        p1 = self.pool1(e1)
        
        e2 = self.enc2(p1)
        p2 = self.pool2(e2)
        
        return p2
    
    def forward(self, x1, x2):
        """Forward pass with two input images"""
        # Encode both images with SHARED weights
        f1 = self.encode(x1)  # April features
        f2 = self.encode(x2)  # July features (same encoder!)
        
        # Concatenate features from both dates
        concat = torch.cat([f1, f2], dim=1)
        
        # Bottleneck
        b = self.bottleneck(concat)
        
        # Decode to change map
        u1 = self.up1(b)
        d1 = self.dec1(u1)
        
        u2 = self.up2(d1)
        d2 = self.dec2(u2)
        
        # Output change probability per pixel
        out = self.output(d2)
        out = self.sigmoid(out)
        
        return out


In [ ]:
# Create model
model = SiameseUNet(in_channels=3).to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())

print(f"✓ Siamese U-Net created")
print(f"  Parameters: {n_params:,}")
print(f"  Device: {device}")
print(f"  Key feature: Shared encoder for both dates")

### Train the Siamese U-Net

**KEY CONCEPTS:**
- **BCE loss** = binary classification (changed vs unchanged pixels)
- **80/20 split** = within left half only (monitor overfitting during training)
- **Mini-batch training** = 8 patch pairs per update
- **Right half** = completely untouched until final evaluation



In [ ]:
# Loss and optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 80/20 split within LEFT HALF patches only
split = int(0.8 * len(X1))
X1_train, X1_val = X1[:split], X1[split:]
X2_train, X2_val = X2[:split], X2[split:]
y_train,  y_val  = y[:split],  y[split:]

print(f"✓ Training setup complete")
print(f"  Train patches: {len(X1_train)} (left half)")
print(f"  Val patches:   {len(X1_val)} (left half)")
print(f"  Right half:    reserved for final test!")

In [ ]:
n_epochs = 10
batch_size = 8

print(f"🏋️ Training for {n_epochs} epochs...")
print("="*60)

losses = []
accuracies = []

model.train()
for epoch in range(n_epochs):
    epoch_loss = 0
    epoch_correct = 0
    epoch_total = 0
    n_batches = 0
    
    for i in range(0, len(X1_train), batch_size):
        batch_x1 = X1_train[i:i+batch_size].to(device)
        batch_x2 = X2_train[i:i+batch_size].to(device)
        batch_y  = y_train[i:i+batch_size].to(device)
        
        # Forward pass through shared encoder
        outputs = model(batch_x1, batch_x2)
        loss = criterion(outputs, batch_y)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
        
        # Track accuracy
        pred_binary = (outputs > 0.5).float()
        epoch_correct += (pred_binary == batch_y).float().sum().item()
        epoch_total += batch_y.numel()
    
    avg_loss = epoch_loss / n_batches
    accuracy = (epoch_correct / epoch_total) * 100
    
    losses.append(avg_loss)
    accuracies.append(accuracy)
    
    print(f"  Epoch {epoch+1:2d}/{n_epochs} | Loss: {avg_loss:.4f} | Accuracy: {accuracy:.1f}%")

print(f"\n✓ Training complete!")
print(f"  Final loss:     {losses[-1]:.4f}")
print(f"  Final accuracy: {accuracies[-1]:.1f}%")

In [ ]:
# Plot loss and accuracy
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, n_epochs+1), losses, linewidth=2, color='steelblue', marker='o')
axes[0].set_xlabel('Epoch', fontweight='bold')
axes[0].set_ylabel('Loss', fontweight='bold')
axes[0].set_title('Training Loss', fontweight='bold')
axes[0].grid(alpha=0.3)

axes[1].plot(range(1, n_epochs+1), accuracies, linewidth=2, color='green', marker='o')
axes[1].set_xlabel('Epoch', fontweight='bold')
axes[1].set_ylabel('Accuracy (%)', fontweight='bold')
axes[1].set_title('Training Accuracy', fontweight='bold')
axes[1].set_ylim([50, 105])
axes[1].axhline(y=100, color='red', linestyle='--', alpha=0.3, label='Perfect')
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.suptitle('Siamese U-Net Training Progress', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Quick validation on left half held-out patches
model.eval()
with torch.no_grad():
    val_preds = model(X1_val.to(device), X2_val.to(device))
    val_preds_binary = (val_preds.cpu().numpy() > 0.5).astype(np.float32)
    val_accuracy = (val_preds_binary == y_val.numpy()).mean()

print(f"✓ Left half validation accuracy: {val_accuracy:.1%}")
print(f"  → Now applying to RIGHT HALF (true generalization test)...")

In [ ]:

# Show 3 random validation examples
fig, axes = plt.subplots(3, 3, figsize=(7, 6))

for i in range(3):
    idx = np.random.randint(0, len(X1_val))
    
    axes[i, 0].imshow(X1_val[idx].permute(1, 2, 0))
    axes[i, 0].set_title('April', fontweight='bold', fontsize=10)
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(y_val[idx].squeeze(), cmap='RdYlGn_r', vmin=0, vmax=1)
    axes[i, 1].set_title('Ground Truth', fontweight='bold', fontsize=10)
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(val_preds_binary[idx].squeeze(), cmap='RdYlGn_r', vmin=0, vmax=1)
    axes[i, 2].set_title('Prediction', fontweight='bold', fontsize=10)
    axes[i, 2].axis('off')

plt.suptitle(f'Validation Examples (Left Half) | Accuracy: {val_accuracy:.1%}', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Apply to Full Image: Sliding Window Prediction

**KEY CONCEPTS:**
- **Sliding window** = systematically moving a fixed-size tile across the image
- **Stride = 64** = 50% overlap between adjacent tiles for smooth predictions
- **Averaging overlaps** = reduces tile boundary artifacts


In [ ]:
# Sliding window prediction on full image
tile_size = 128
stride = 64
h, w = image_april.shape[:2]


# Pad images so tiles cover full extent
pad = tile_size // 2
april_padded = np.pad(image_april, ((pad,pad),(pad,pad),(0,0)), mode='reflect')
july_padded  = np.pad(image_july,  ((pad,pad),(pad,pad),(0,0)), mode='reflect')

change_map_unet = np.zeros((h + 2*pad, w + 2*pad), dtype=np.float32)
count_map       = np.zeros((h + 2*pad, w + 2*pad), dtype=np.float32)


model.eval()
with torch.no_grad():
    for y_pos in range(0, h + 2*pad - tile_size + 1, stride):
        for x_pos in range(0, w + 2*pad - tile_size + 1, stride):
            t1 = torch.FloatTensor(april_padded[y_pos:y_pos+tile_size, x_pos:x_pos+tile_size]).permute(2,0,1).unsqueeze(0).to(device)
            t2 = torch.FloatTensor(july_padded[y_pos:y_pos+tile_size,  x_pos:x_pos+tile_size]).permute(2,0,1).unsqueeze(0).to(device)
            
            pred = model(t1, t2).squeeze().cpu().numpy()
            change_map_unet[y_pos:y_pos+tile_size, x_pos:x_pos+tile_size] += pred
            count_map[y_pos:y_pos+tile_size,       x_pos:x_pos+tile_size] += 1
        
        print(f"  Processing: {(y_pos/(h+2*pad-tile_size))*100:.0f}%", end='\r')


In [ ]:
# Average and crop back to original size
change_map_unet = (change_map_unet / (count_map + 1e-6))[pad:pad+h, pad:pad+w]

print(f"\n✓ Full image processed: {change_map_unet.shape}")
print(f"  No black borders - full coverage!")

In [ ]:
# Binary prediction map
change_map_binary = (change_map_unet > 0.5).astype(np.float32)
split_col = w // 2


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Probability map with split line
axes[0].imshow(change_map_unet, cmap='hot', vmin=0, vmax=1)
axes[0].axvline(x=split_col, color='cyan', linewidth=2, linestyle='--')
axes[0].set_title('Change Probability Map\n(0=No change, 1=Change)', fontweight='bold', fontsize=12)
axes[0].axis('off')

# Binary prediction with split line
axes[1].imshow(change_map_binary, cmap='RdYlGn_r')
axes[1].axvline(x=split_col, color='cyan', linewidth=2, linestyle='--')
axes[1].set_title(f'Binary Prediction (threshold=0.5)\n{change_map_binary.sum()/change_map_binary.size*100:.1f}% changed', 
                  fontweight='bold', fontsize=12)
axes[1].axis('off')

# Overlay on July RGB
axes[2].imshow(image_july)
overlay = np.zeros((*change_map_binary.shape, 4))
overlay[change_map_binary == 1] = [1, 0, 0, 0.5]
axes[2].imshow(overlay)
axes[2].axvline(x=split_col, color='cyan', linewidth=2, linestyle='--')
axes[2].set_title('Detected Changes on July RGB\n(Red=Change, Cyan=Train/Test split)', 
                  fontweight='bold', fontsize=12)
axes[2].axis('off')

plt.suptitle('Siamese U-Net Change Detection: Full Scene', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Calculate metrics separately for left (trained) and right (unseen) halves
change_binary_adj = change_binary[:change_map_binary.shape[0], :change_map_binary.shape[1]]

left_pred  = change_map_binary[:, :split_col].flatten()
left_truth = change_binary_adj[:, :split_col].flatten()
right_pred  = change_map_binary[:, split_col:].flatten()
right_truth = change_binary_adj[:, split_col:].flatten()

from sklearn.metrics import jaccard_score, f1_score

left_iou  = jaccard_score(left_truth,  left_pred)
right_iou = jaccard_score(right_truth, right_pred)
left_f1   = f1_score(left_truth,  left_pred)
right_f1  = f1_score(right_truth, right_pred)

print("="*60)
print("SIAMESE U-NET CHANGE DETECTION RESULTS")
print("="*60)
print(f"\n📊 LEFT HALF (trained region):")
print(f"  IoU: {left_iou:.3f} | F1: {left_f1:.3f}")
print(f"\n📊 RIGHT HALF (unseen region - true test):")
print(f"  IoU: {right_iou:.3f} | F1: {right_f1:.3f}")
print(f"\n  Generalization gap: {abs(left_iou-right_iou):.3f} IoU points")
print(f"\n✅ Advantages over NDVI Differencing:")
print(f"  • Spatial context (not just pixel values)")
print(f"  • Learned semantic changes")
print(f"  • Smooth probability map (not binary threshold)")
print("="*60)